# Coordinate-grid generation

`Nematics3D` provides two small utilities for constructing regular coordinate grids:

- `nematics3d.generate_coordinate_grid()` maps a target grid onto the index space of a source array.
- `nematics3d.generate_fixed_step_grid()` constructs a two-dimensional grid from requested physical sizes and step lengths.

The first is useful for regular resampling and coordinate transforms. The second is used when a plane should be sampled at approximately fixed physical spacing.


## Setup

**For readers who are only interested in the tutorial, this section can be safely skipped.** The following cell imports `NumPy` and `Nematics3D`.


In [ ]:
import numpy as np
import nematics3d as n3d


## `generate_coordinate_grid()`

`generate_coordinate_grid(shape_source, shape_target)` returns a floating-point coordinate array with shape `(*shape_target, ndim)`. Each entry gives the position of one target-grid point in the source array's index space. The target grid spans the complete source-index interval along every axis.

For example, resampling a one-dimensional source of length 5 onto 4 target points gives source coordinates $0$, $4/3$, $8/3$, and $4$.


In [ ]:
grid_1d = n3d.generate_coordinate_grid((5,), (4,))
print(grid_1d)
print("shape:", grid_1d.shape)


### Two-dimensional example

For a source shape `(5, 7)` and target shape `(3, 4)`, the first target axis spans source coordinates from 0 to 4, while the second spans 0 to 6. The final axis of the returned array stores the two source-index coordinates.


In [ ]:
grid_2d = n3d.generate_coordinate_grid((5, 7), (3, 4))
print("shape:", grid_2d.shape)
print(grid_2d)


### Identical source and target shapes

When the two shapes are identical, the returned coordinates are the ordinary array indices represented as floating-point values. This is useful when index coordinates will subsequently be mapped into physical space by a grid transform.


In [ ]:
grid_identity = n3d.generate_coordinate_grid((2, 3), (2, 3))
print(grid_identity)


### Inputs and edge cases

Both shapes must be non-empty sequences of positive integers and must have the same number of dimensions. One-, two-, three-, and higher-dimensional grids are supported. Boolean dimensions, floating-point dimensions, zero, and negative dimensions are rejected.

If a target dimension has length one, that axis samples source coordinate zero. This follows the convention of `numpy.linspace(0, source_size - 1, 1)`.

The function returns only the dense source-coordinate grid. Integer target-grid indices and target spacing are not constructed because they are not required by the current public use cases and can be derived separately when needed.


## `generate_fixed_step_grid()`

`generate_fixed_step_grid(size1, size2, step1, step2, alignment=...)` creates a regular two-dimensional grid. Unlike `generate_coordinate_grid()`, which is specified by source and target array shapes, this function is specified by requested coordinate extents and step lengths.

It returns three values:

1. `grid`: floating-point coordinates with shape `(n1, n2, 2)`.
2. `grid_int`: integer grid indices with the same leading shape.
3. `size_eff`: the actual coordinate extent covered by the discrete grid.

The effective size can be smaller than the requested size because the function uses only complete steps.


### Bottom-left alignment

The default `alignment="bottom-left"` places the first grid point at `(0, 0)` and extends in the positive coordinate directions.


In [ ]:
grid, grid_int, size_eff = n3d.generate_fixed_step_grid(2.5, 2.0, 1.0, 0.75)
print("coordinates:\n", grid)
print("integer indices:\n", grid_int)
print("effective size:", size_eff)


### Center alignment

With `alignment="center"`, the grid contains the origin and extends symmetrically in both directions. The number of samples along each axis is therefore odd.


In [ ]:
grid_center, grid_int_center, size_eff_center = n3d.generate_fixed_step_grid(
    5.0, 3.0, 1.0, 1.0, alignment="center"
)
print("coordinates:\n", grid_center)
print("integer indices:\n", grid_int_center)
print("effective size:", size_eff_center)


### Input rules

`size1` and `size2` must be finite, non-negative real numbers. `step1` and `step2` must be at least `1e-12` and finite. The supported alignments are exactly `"bottom-left"` and `"center"`.

A requested size of zero is valid and produces one grid point along that axis. A requested size does not need to be an integer multiple of the corresponding step; `size_eff` reports the extent actually covered.


## Which function should I use?

| Goal | Function |
| --- | --- |
| Resample an existing array onto a chosen target shape | `generate_coordinate_grid()` |
| Construct index coordinates before applying a physical grid transform | `generate_coordinate_grid()` |
| Sample a two-dimensional plane with approximately fixed spacing | `generate_fixed_step_grid()` |
| Need integer topology for a generated two-dimensional plane | `generate_fixed_step_grid()` |

The two functions therefore describe different ways of specifying a regular grid: one fixes the number of samples, while the other fixes the desired spacing.


## Implementation notes

**This section is intended for developers. Regular users can safely skip it.**

`generate_coordinate_grid()` allocates only the returned dense coordinate array and one small one-dimensional coordinate vector at a time during general resampling. When source and target shapes are identical, it uses `numpy.indices()` directly. It deliberately does not construct an unused dense integer-coordinate grid.

`generate_fixed_step_grid()` constructs the integer topology once with `numpy.indices()` and derives the floating coordinates from it. This avoids building separate `numpy.meshgrid()` intermediates for the integer and floating representations.
